# 🏠 Furniture & Home Décor — Rental Data Generator
> **Notebook 1B · Furniture** — Drop-in replacement for `01_data_generation.ipynb`. Same output file names and CSV schema, so the EDA, A/B testing, and ML notebooks work unchanged.
> Outputs: `../data/generated_data/*.csv` · MySQL write when credentials are available

## 0 · Imports & Connection

> Same setup as the electronics notebook — connects to MySQL if credentials are present, falls back to CSV-only mode if not. The `save()` function always writes a CSV; MySQL is a bonus.
> The try/except blocks aren't defensive coding for its own sake — they let this notebook run in any environment without crashing, including a fresh machine without a database.

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
from pathlib import Path

try:
    from sqlalchemy import create_engine, text
except Exception:
    create_engine = None
    text = None

try:
    from dotenv import load_dotenv
except Exception:
    def load_dotenv():
        return None

load_dotenv()
np.random.seed(42)

DATA_DIR    = "../data/generated_data"
TABLEAU_DIR = "../data/tableau"
FIGURES_DIR = "../figures"
SQL_DIR     = "../data/sql"

for d in [DATA_DIR, TABLEAU_DIR, FIGURES_DIR, SQL_DIR]:
    os.makedirs(d, exist_ok=True)

engine = None
if create_engine is not None and os.getenv("DB_USER") and os.getenv("DB_PASSWORD") and os.getenv("DB_HOST") and os.getenv("DB_NAME"):
    try:
        engine = create_engine(
            f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
            f"@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}",
            echo=False
        )
        with engine.begin() as conn:
            conn.execute(text("SET FOREIGN_KEY_CHECKS = 0"))
            for t in ["return_conditions", "inventory_events", "rentals",
                      "rental_revenue_vs_discount", "customers", "pricing_rules",
                      "products", "categories"]:
                conn.execute(text(f"DROP TABLE IF EXISTS `{t}`"))
            conn.execute(text("SET FOREIGN_KEY_CHECKS = 1"))
        print("MySQL connection OK. Tables will be refreshed.")
    except Exception as e:
        print(f"MySQL unavailable. CSV-only mode. Reason: {e}")
        engine = None
else:
    print("MySQL credentials not found. CSV-only mode.")

def save(name, df):
    """Write DataFrame to CSV and, when available, to MySQL."""
    csv_path = f"{DATA_DIR}/{name}.csv"
    df.to_csv(csv_path, index=False)
    if engine is not None:
        df.to_sql(name, engine, if_exists="replace", index=False)
    print(f"  {name}: {len(df):,} rows")

MySQL connection OK. Tables will be refreshed.


## 1 · Categories

This is an IKEA-led programme applied to the furniture and home décor market.

The same three fields from the electronics notebook drive everything downstream:
- `depreciation_class` — how fast the item loses resale value. Furniture holds value better than electronics: a MALM bed frame from 2022 still sells secondhand at ~70% of retail. The `slow` class reflects this.
- `rental_demand_tier` — how frequently renters actually want this category. Sofas, beds, and office furniture are high-demand because people need them immediately when moving. Lighting and outdoor furniture are medium — useful but not urgent.
- `rental_programme` — whether this category makes sense to rent at all.

**4 categories excluded:**
- **Outdoor Decor** — garden ornaments, plant pots: too low value, weather-exposed
- **Wall Art & Mirrors** — highly personal taste; customers rarely want someone else's art
- **Rugs & Textiles** — hygiene and wear concerns; customers don't want used fabric items
- **Small Accessories** — candles, frames, vases: too cheap for the programme overhead

**Depreciation notes:**
- `slow` (Sofas, Beds, Dining, Storage, Designer): IKEA flat-pack holds value well. Standardised parts are replaceable. 5–6% annual depreciation is consistent with IKEA AS-IS resale data.
- `standard` (Office, Outdoor, Lighting, Home Decor): moderate wear from regular use. 8–10%/yr.
- `fast` (Kids & Nursery, Event Furniture): outgrown quickly or used hard commercially. 13–14%/yr.

In [2]:
categories_data = [
    # FIX: Kids & Nursery moved to 'fast' depreciation — IKEA Sniglar/Stuva outgrown quickly;
    #      Event Furniture moved to 'fast' — high-turnover commercial use.
    #      Designer Statement Pieces kept 'slow' — Vitra/Knoll hold value well.
    {"category_id": 1,  "category_name": "Sofas & Seating",          "depreciation_class": "slow",     "avg_depreciation_rate": 0.06, "rental_demand_tier": "high",   "rental_programme": True},
    {"category_id": 2,  "category_name": "Beds & Mattresses",        "depreciation_class": "slow",     "avg_depreciation_rate": 0.05, "rental_demand_tier": "high",   "rental_programme": True},
    {"category_id": 3,  "category_name": "Dining Furniture",         "depreciation_class": "slow",     "avg_depreciation_rate": 0.05, "rental_demand_tier": "medium", "rental_programme": True},
    {"category_id": 4,  "category_name": "Office Furniture",         "depreciation_class": "standard", "avg_depreciation_rate": 0.08, "rental_demand_tier": "high",   "rental_programme": True},
    {"category_id": 5,  "category_name": "Storage & Cabinets",       "depreciation_class": "slow",     "avg_depreciation_rate": 0.05, "rental_demand_tier": "medium", "rental_programme": True},
    {"category_id": 6,  "category_name": "Home Decor",               "depreciation_class": "standard", "avg_depreciation_rate": 0.10, "rental_demand_tier": "medium", "rental_programme": True},
    {"category_id": 7,  "category_name": "Outdoor Furniture",        "depreciation_class": "standard", "avg_depreciation_rate": 0.09, "rental_demand_tier": "medium", "rental_programme": True},
    {"category_id": 8,  "category_name": "Lighting",                 "depreciation_class": "standard", "avg_depreciation_rate": 0.08, "rental_demand_tier": "medium", "rental_programme": True},
    {"category_id": 9,  "category_name": "Kids & Nursery",           "depreciation_class": "fast",     "avg_depreciation_rate": 0.13, "rental_demand_tier": "high",   "rental_programme": True},
    {"category_id": 10, "category_name": "Event Furniture",          "depreciation_class": "fast",     "avg_depreciation_rate": 0.14, "rental_demand_tier": "high",   "rental_programme": True},
    {"category_id": 11, "category_name": "Outdoor Decor",            "depreciation_class": "slow",     "avg_depreciation_rate": 0.07, "rental_demand_tier": "low",    "rental_programme": False},
    {"category_id": 12, "category_name": "Wall Art & Mirrors",       "depreciation_class": "slow",     "avg_depreciation_rate": 0.06, "rental_demand_tier": "low",    "rental_programme": False},
    {"category_id": 13, "category_name": "Rugs & Textiles",          "depreciation_class": "standard", "avg_depreciation_rate": 0.09, "rental_demand_tier": "low",    "rental_programme": False},
    {"category_id": 14, "category_name": "Small Accessories",        "depreciation_class": "standard", "avg_depreciation_rate": 0.10, "rental_demand_tier": "low",    "rental_programme": False},
    {"category_id": 15, "category_name": "Designer Statement Pieces","depreciation_class": "slow",     "avg_depreciation_rate": 0.05, "rental_demand_tier": "high",   "rental_programme": True},
]
categories = pd.DataFrame(categories_data)
save("categories", categories)

  categories: 15 rows


## 2 · Pricing Rules

Same A/B structure as the electronics notebook — two pricing models, three duration tiers, two experiment groups.

**Two key differences from electronics:**

1. **Duration models are longer.** Electronics used `7_day / 30_day / flexible`. Furniture uses `30_day / 90_day / flexible` — nobody rents a sofa for a week. The minimum on `30_day` is 30 days; on `90_day` it's 90 days; `flexible` starts at 7 days for event/short-term use.

2. **Rate floors are higher.** Real-world benchmarks (IKEA FaaS pilot, Feather, Furlenco):
   - Typical monthly rate: 3–8% of retail value
   - Flat-rate floor raised to €4.50/day — furniture requires delivery, assembly, and collection, which electronics don't
   - `pct_of_retail_daily` floor raised to 0.0010 (from 0.0007 in electronics) to stay above the delivery cost threshold

In [3]:
pricing_data = []
rule_id = 1
for pricing_model in ["flat_rate", "pct_of_retail"]:
    for duration_model in ["30_day", "90_day", "flexible"]:
        for experiment_group in ["A", "B"]:
            if pricing_model == "flat_rate":
                # FIX: floor raised from 2.2 to 4.5 — furniture has real delivery/handling costs
                base_daily = np.random.uniform(4.5, 12.5)
                # FIX: floor raised from 0.0007 to 0.0010 — matches Feather/Furlenco benchmarks
                pct_daily = np.random.uniform(0.0010, 0.0022)
            else:
                base_daily = np.random.uniform(3.5, 10.0)
                # FIX: ceiling raised from 0.0023 to 0.0028 — pct_of_retail model earns more on premium items
                pct_daily = np.random.uniform(0.0012, 0.0028)
            pricing_data.append({
                "rule_id":              rule_id,
                "pricing_model":        pricing_model,
                "duration_model":       duration_model,
                "experiment_group":     experiment_group,
                "base_daily_rate":      round(base_daily, 2),
                "pct_of_retail_daily":  round(pct_daily, 4),
                "min_rental_days":      7 if duration_model == "flexible" else (30 if duration_model == "30_day" else 90),
                "max_rental_days":      180,
                "late_fee_per_day":     round(np.random.uniform(3, 12), 2),
                "security_deposit_pct": round(np.random.uniform(0.10, 0.22), 2),
                "insurance_fee_pct":    round(np.random.uniform(0.015, 0.045), 3),
                "created_at":           "2021-01-01",
            })
            rule_id += 1

pricing = pd.DataFrame(pricing_data)
save("pricing_rules", pricing)

  pricing_rules: 12 rows


## 3 · Seasonal Demand

Furniture rental seasonality is driven by life events, not gifting seasons — which is the key difference from electronics.

**Why these peaks:**
- **May/June (1.12–1.28×):** Student end-of-year moves, expat relocations arriving in Lisbon and Madrid for summer, short-term tenancy starts. This is the biggest driver of sofa and bed rentals.
- **September (1.08–1.18×):** Back-to-university setups, corporate office refreshes (IKEA Business accounts), new tenancy cycles starting in autumn.
- **November/December (1.15–1.22×):** Holiday hosting prep, interior refresh before Christmas guests, gifting of furniture subscriptions.
- **Event Furniture (cat 10):** Spring and summer wedding + corporate event season dominates — hence HIGH_SEASON_CATS includes cat 10.

In [4]:
SEASONAL_STD  = {1:0.82,2:0.82,3:0.90,4:1.00,5:1.12,6:1.15,7:0.98,8:0.92,9:1.08,10:1.05,11:1.18,12:1.22}
SEASONAL_HIGH = {1:0.78,2:0.80,3:0.88,4:1.02,5:1.20,6:1.28,7:1.05,8:0.95,9:1.18,10:1.08,11:1.15,12:1.20}
SEASONAL_LOW  = {1:0.92,2:0.92,3:0.96,4:1.00,5:1.03,6:1.05,7:0.98,8:0.95,9:1.00,10:1.03,11:1.08,12:1.10}
HIGH_SEASON_CATS = {1, 2, 4, 9, 10, 15}

def get_seasonal_table(cat_id, demand_tier):
    if cat_id in HIGH_SEASON_CATS:
        return SEASONAL_HIGH
    elif demand_tier == "low":
        return SEASONAL_LOW
    return SEASONAL_STD

print("Seasonal tables defined.")

Seasonal tables defined.


## 4 · Products

620 products across 15 categories. IKEA gets 55–68% brand weight across programme categories — this reflects the IKEA-as-a-Service framing. Competitor brands (Article, Made, West Elm, Steelcase) form the realistic minority tail.

**Key design decisions:**
- `PROG_END = 2024-12-31` — fixed snapshot date, same as electronics. All aging and comparison logic is relative to this date.
- `rental_eligible_date = listed_date + 365 days` — same rule as electronics: one full year unsold before the item enters the programme.
- Real IKEA product names (KIVIK, MALM, PAX, BEKANT, etc.) — makes the dataset immediately recognisable and credible in a presentation context.
- `suffixes` are colour/material variants (white, oak, beige) — realistic for IKEA's naming system.
- Designer Statement Pieces (cat 15) use Vitra, Cassina, Knoll — no IKEA; price range €900–€12,000. These are the high-ratio anchors of the dataset.

In [5]:
# FIX: IKEA dominant in all relevant categories — reflects IKEA-as-a-Service programme framing
brands_by_cat = {
    1:  [("IKEA", 0.60), ("Article", 0.15), ("Made", 0.13), ("Habitat", 0.07), ("West Elm", 0.05)],
    2:  [("IKEA", 0.65), ("Emma", 0.12), ("Simba", 0.10), ("Habitat", 0.08), ("Wayfair", 0.05)],
    3:  [("IKEA", 0.62), ("Made", 0.14), ("West Elm", 0.12), ("Kave Home", 0.07), ("La Redoute", 0.05)],
    4:  [("IKEA", 0.55), ("Steelcase", 0.18), ("Herman Miller", 0.14), ("Flexispot", 0.08), ("Branch", 0.05)],
    5:  [("IKEA", 0.68), ("Habitat", 0.12), ("Kave Home", 0.10), ("BoConcept", 0.06), ("Wayfair", 0.04)],
    6:  [("IKEA", 0.58), ("H&M Home", 0.16), ("Zara Home", 0.14), ("West Elm", 0.08), ("La Redoute", 0.04)],
    7:  [("IKEA", 0.60), ("Kettler", 0.16), ("Kave Home", 0.12), ("Outsunny", 0.08), ("Habitat", 0.04)],
    8:  [("IKEA", 0.62), ("Philips", 0.16), ("Flos", 0.10), ("Habitat", 0.07), ("West Elm", 0.05)],
    9:  [("IKEA", 0.65), ("Stokke", 0.14), ("Snuz", 0.10), ("Mamas & Papas", 0.07), ("Babyletto", 0.04)],
    10: [("IKEA", 0.60), ("EventSource", 0.18), ("Kave Home", 0.12), ("Habitat", 0.06), ("Wayfair", 0.04)],
    11: [("IKEA", 0.65), ("Kettler", 0.18), ("West Elm", 0.10), ("Habitat", 0.07)],
    12: [("IKEA", 0.55), ("Desenio", 0.22), ("Habitat", 0.13), ("West Elm", 0.10)],
    13: [("IKEA", 0.60), ("Lorena Canals", 0.20), ("Habitat", 0.12), ("La Redoute", 0.08)],
    14: [("IKEA", 0.65), ("Zara Home", 0.18), ("H&M Home", 0.12), ("Habitat", 0.05)],
    15: [("Vitra", 0.30), ("Cassina", 0.25), ("Knoll", 0.20), ("B&B Italia", 0.15), ("Muuto", 0.10)],
}

# IKEA product name roots — real IKEA naming style (Swedish-inspired, often single word)
name_roots_by_cat = {
    1:  ["KIVIK", "SÖDERHAMN", "EKTORP", "MORABO", "LANDSKRONA", "UPPLAND", "ÄPPLARYD"],
    2:  ["MALM", "HEMNES", "BJÖRKSNAS", "SAGSTUA", "NEIDEN", "SLATTUM", "TARVA"],
    3:  ["LISABO", "EKEDALEN", "INGATORP", "NORDVIKEN", "MÖRBYLÅNGA", "STORNÄS"],
    4:  ["BEKANT", "MITTZON", "FREDDE", "MICKE", "TROTTEN", "UPPSPEL", "MARKUS"],
    5:  ["PAX", "BRIMNES", "KALLAX", "IVAR", "BILLY", "HAVSTA", "LIXHULT"],
    6:  ["FEJKA", "SMYCKA", "SKUGGIS", "SÖNDRUM", "STRALA", "VINTER"],
    7:  ["ÄPPLARÖ", "FALHOLMEN", "HUSARÖ", "TÄRNÖ", "FROSON", "SOLVINDEN"],
    8:  ["HEKTAR", "HOLMO", "RANARP", "LERSTA", "SKURUP", "TERTIAL", "NYMÅNE"],
    9:  ["SNIGLAR", "STUVA", "SUNDVIK", "GULLIVER", "KURA", "BUSUNGE", "SMÅGÖRA"],
    10: ["TÄRENDÖ", "NORDEN", "OPPEBY", "LISABO", "TOMMARYD", "GUNDE"],
    11: ["BORRBY", "SOCKER", "INGEFÄRA", "STICKAT"],
    12: ["RIBBA", "MOSSLANDA", "SANNAHED", "KNOPPÄNG"],
    13: ["STOENSE", "PERSISK", "SORTSÖ", "TIPHEDE"],
    14: ["ENSIDIG", "SOCKERKAKA", "DRÖMSÄCK", "KLIPPKAKTUS"],
    15: ["Vitra Eames Chair", "Cassina LC4", "Knoll Barcelona", "B&B Italia Tufty", "Muuto Outline"],
}

suffixes = ["white", "black", "oak", "birch", "pine", "grey", "beige", "walnut", ""]
PROG_END = datetime(2024, 12, 31)

n_per_cat = [
    65,  # Sofas & Seating
    55,  # Beds & Mattresses
    45,  # Dining Furniture
    50,  # Office Furniture
    45,  # Storage & Cabinets
    55,  # Home Decor
    35,  # Outdoor Furniture
    40,  # Lighting
    35,  # Kids & Nursery
    40,  # Event Furniture
    25,  # Outdoor Decor
    30,  # Wall Art & Mirrors
    35,  # Rugs & Textiles
    45,  # Small Accessories
    20,  # Designer Statement Pieces
]

price_bands_by_cat = {
    1:  [(180, 450, 0.30), (450, 900, 0.45), (900, 1800, 0.20), (1800, 3200, 0.05)],
    2:  [(220, 500, 0.25), (500, 1000, 0.45), (1000, 1800, 0.25), (1800, 3000, 0.05)],
    3:  [(120, 300, 0.30), (300, 700, 0.45), (700, 1400, 0.20), (1400, 2600, 0.05)],
    4:  [(120, 280, 0.25), (280, 650, 0.45), (650, 1400, 0.25), (1400, 2400, 0.05)],
    5:  [(90, 250, 0.30), (250, 600, 0.45), (600, 1200, 0.20), (1200, 2200, 0.05)],
    6:  [(20, 80, 0.35), (80, 180, 0.40), (180, 350, 0.20), (350, 700, 0.05)],
    7:  [(140, 350, 0.30), (350, 800, 0.45), (800, 1600, 0.20), (1600, 2800, 0.05)],
    8:  [(30, 90, 0.30), (90, 220, 0.45), (220, 500, 0.20), (500, 1200, 0.05)],
    9:  [(80, 220, 0.35), (220, 500, 0.40), (500, 1000, 0.20), (1000, 1800, 0.05)],
    10: [(60, 180, 0.30), (180, 450, 0.45), (450, 900, 0.20), (900, 1600, 0.05)],
    11: [(40, 110, 0.35), (110, 240, 0.40), (240, 500, 0.20), (500, 900, 0.05)],
    12: [(50, 150, 0.35), (150, 320, 0.40), (320, 700, 0.20), (700, 1400, 0.05)],
    13: [(40, 120, 0.30), (120, 280, 0.45), (280, 550, 0.20), (550, 1000, 0.05)],
    14: [(10, 35, 0.35), (35, 80, 0.40), (80, 160, 0.20), (160, 350, 0.05)],
    15: [(900, 1800, 0.25), (1800, 3500, 0.45), (3500, 7000, 0.25), (7000, 12000, 0.05)],
}

def random_listed_date():
    year = np.random.choice([2020, 2021, 2022, 2023, 2024], p=[0.02, 0.08, 0.22, 0.36, 0.32])
    if year == 2024:
        return datetime(2024, 1, 1) + timedelta(days=int(np.random.uniform(0, 181)))
    return datetime(year, 1, 1) + timedelta(days=int(np.random.uniform(0, 365)))

def sample_retail_price(category_id):
    bands = price_bands_by_cat[category_id]
    probs = [b[2] for b in bands]
    idx = np.random.choice(range(len(bands)), p=probs)
    low, high, _ = bands[idx]
    return round(np.random.uniform(low, high), 2)

def sample_brand(cat_id):
    """Sample brand with IKEA-dominant weighted distribution."""
    brand_weights = brands_by_cat[cat_id]
    brands = [b[0] for b in brand_weights]
    probs  = [b[1] for b in brand_weights]
    return np.random.choice(brands, p=probs)

products_list = []
pid = 1

for cat in categories_data:
    cid = cat["category_id"]
    for _ in range(n_per_cat[cid - 1]):
        retail = sample_retail_price(cid)
        listed = random_listed_date()
        # FIX: 300 -> 365 days — items enter programme after exactly 1 year unsold
        elig = listed + timedelta(days=365)
        yrs = max(0, (PROG_END - listed).days / 365)
        dep = max(0.03, min(cat["avg_depreciation_rate"] + np.random.normal(0, 0.015), 0.22))
        selected_brand = sample_brand(cid)
        root = np.random.choice(name_roots_by_cat[cid])
        suffix = np.random.choice(suffixes)
        product_name = f"{selected_brand} {root} {suffix}".strip() if selected_brand not in root else f"{root} {suffix}".strip()
        products_list.append({
            "product_id":               pid,
            "category_id":              cid,
            "product_name":             product_name,
            "brand":                    selected_brand,
            "original_retail_price":    retail,
            "current_depreciated_value": round(retail * max(0.18, 1 - dep * yrs), 2),
            "condition_grade":          np.random.choice(["A", "B", "C"], p=[0.55, 0.32, 0.13]),
            "listed_date":              listed.date(),
            "rental_eligible_date":     elig.date(),
            # FIX: Added IKEA Online as 4th retailer — reflects actual IKEA PT/ES distribution
            "retailer":                 np.random.choice(
                ["IKEA Portugal", "IKEA Spain", "IKEA Online", "Westwing ES"],
                p=[0.42, 0.28, 0.18, 0.12]
            ),
            "is_active": 1,
        })
        pid += 1

products = pd.DataFrame(products_list)
save("products", products)

  products: 620 rows


## 5 · Customers

2,000 customers, same PT/ES split as electronics (56% PT, 44% ES). Segment weights are adjusted for the furniture market:

- `professional` (34%) — highest weight: remote workers furnishing home offices, expats setting up apartments
- `casual` (30%) — second highest: people furnishing short-term rentals or between moves
- `business` (18%) — corporate office setups, co-working spaces, event companies
- `student` (18%) — lower than electronics; students rent furniture but less frequently than tech products

The segment weights shift slightly from electronics (student was 25%) because furniture rental skews older — students are less likely to invest in a sofa subscription.

In [6]:
first_names = ["Ana","Pedro","Maria","João","Sofia","Miguel","Inês","Ricardo","Beatriz","Tiago",
               "Carlos","Luísa","Fernando","Catarina","André","Marta","Rui","Sara","Diogo","Filipa",
               "Elena","Marco","Lucia","Pablo","Rosa","Diego","Carmen","Rafael","Isabel","Nuno"]
last_names  = ["Silva","Santos","Ferreira","Pereira","Costa","Oliveira","Rodrigues","Martins",
               "Jesus","Sousa","Fernández","García","López","Martínez","González","Sánchez"]
cities_pt   = ["Lisboa","Porto","Braga","Coimbra","Setúbal","Faro","Évora","Aveiro","Funchal","Leiria"]
cities_es   = ["Madrid","Barcelona","Valencia","Sevilla","Zaragoza","Málaga","Bilbao","Alicante"]
segments    = ["student","professional","business","casual"]
seg_w       = [0.18, 0.34, 0.18, 0.30]

customers_list = []
for cid in range(1, 2001):
    country = np.random.choice(["PT","ES"], p=[0.56, 0.44])
    reg = datetime(2021,1,1) + timedelta(days=int(np.random.uniform(0, 365*2)))
    customers_list.append({
        "customer_id":       cid,
        "first_name":        np.random.choice(first_names),
        "last_name":         np.random.choice(last_names),
        "city":              np.random.choice(cities_pt if country=="PT" else cities_es),
        "country":           country,
        "customer_segment":  np.random.choice(segments, p=seg_w),
        "registration_date": reg.date(),
    })
customers = pd.DataFrame(customers_list)
save("customers", customers)

  customers: 2,000 rows


## 6 · Customer Repeat Rental Pool

Same weighted pool mechanism as the electronics notebook. Each customer gets a number of "slots" based on their segment, creating realistic repeat rental behaviour.

**Furniture-specific adjustments to `SEG_MONTH_BOOST`:**
- Students spike in **September** (1.55×) — back-to-university furniture rush
- Students also spike in **August** (1.25×) — moving in early before term
- Casual customers peak in **May–June** (1.15–1.20×) — summer tenancy moves
- Professionals boost slightly in **May–June** (1.08–1.10×) and September (1.12×) — corporate relocations
- Business customers peak in **March–April** (1.08–1.12×) and September (1.10×) — office setup cycles

These patterns mirror real Iberian tenancy and corporate cycles.

In [7]:
SEG_RENTAL_DIST = {
    "business":     ([3,4,5,6,7,8], [0.08,0.18,0.25,0.22,0.17,0.10]),
    "professional": ([2,3,4,5],     [0.22,0.35,0.28,0.15]),
    "student":      ([1,2,3],       [0.42,0.38,0.20]),
    "casual":       ([1,2,3],       [0.52,0.34,0.14]),
}
SEG_MONTH_BOOST = {
    "student":      {1:1.10,2:1.05,8:1.25,9:1.55,10:1.18},
    "casual":       {5:1.15,6:1.20,11:1.20,12:1.25},
    "professional": {5:1.08,6:1.10,9:1.12},
    "business":     {3:1.08,4:1.12,9:1.10},
}

customer_pool = []
for _, row in customers.iterrows():
    vals, probs = SEG_RENTAL_DIST[row["customer_segment"]]
    n = int(np.random.choice(vals, p=probs))
    customer_pool.extend([row["customer_id"]] * n)
customer_pool = np.array(customer_pool)
np.random.shuffle(customer_pool)
pool_idx = 0
customer_segment_map = customers.set_index("customer_id")["customer_segment"].to_dict()

def next_customer(month=None):
    global pool_idx
    for _ in range(8):
        if pool_idx >= len(customer_pool):
            pool_idx = 0
            np.random.shuffle(customer_pool)
        cid = int(customer_pool[pool_idx]); pool_idx += 1
        if month is None:
            return cid
        seg   = customer_segment_map.get(cid, "casual")
        boost = SEG_MONTH_BOOST.get(seg, {}).get(month, 1.0)
        if np.random.random() < boost / 1.55:
            return cid
    if pool_idx >= len(customer_pool):
        pool_idx = 0
    cid = int(customer_pool[pool_idx]); pool_idx += 1
    return cid

print(f"Customer pool: {len(customer_pool):,} slots")

Customer pool: 5,889 slots


## 7 · Rentals, Returns & Inventory Events

Same structure as the electronics rental loop, with three furniture-specific adaptations:

**1. Duration is longer.** Furniture rentals run 30–120 days for high-demand items; 14–90 days for medium; 14–60 days for low. Event furniture (cat 10) is the exception — short bursts of 3–21 days for weddings, launches, pop-ups.

**2. `choose_duration()` and `choose_rental_count()` are extracted as functions.** This keeps the main loop readable. In electronics, duration logic was inline — it worked but was harder to follow. Extracting it makes the business rules explicit.

**3. Operational costs are higher and tiered by category.** Furniture has real delivery, assembly, and collection costs that electronics don't. Heavy items (sofas, beds, dining sets, office furniture, outdoor, event, designer) get 22–35% ops cost. Lighter items get 16–28%. Source: Feather ~28–32%, Furlenco ~25–30%, IKEA FaaS pilot est. 30–35%.

**4. No-return rate is lower (2.5% vs 5.5%).** Furniture is hard to disappear with — it's bulky, tracked, and the customer paid a deposit. Electronics can walk out more easily.

In [8]:
rentals_list = []
returns_list = []
events_list  = []
rid = 1

def choose_duration(cat_id, demand, month):
    # Furniture rentals are longer than electronics — IKEA FaaS targets 1–6 month cycles
    if cat_id == 10:  # event furniture: short bursts
        dur = int(np.random.choice([3, 7, 14, 21], p=[0.20, 0.40, 0.28, 0.12]))
    elif cat_id in [1,2,4,5,9,15] or demand == "high":
        dur = int(np.random.choice([30,60,90,120], p=[0.28,0.34,0.24,0.14]))
    elif demand == "medium":
        dur = int(np.random.choice([14,30,60,90], p=[0.18,0.36,0.30,0.16]))
    else:
        dur = int(np.random.choice([14,30,45,60], p=[0.20,0.40,0.25,0.15]))
    if month in (5,6,9) and cat_id in [1,2,4,9]:
        dur = int(np.random.choice([30,60,90], p=[0.22,0.46,0.32]))
    return dur

def choose_rental_count(cat_id, demand, days_available):
    max_possible = max(1, days_available // 21)
    if cat_id == 10:
        base = np.random.choice([7,8,9,10,11], p=[0.12,0.24,0.30,0.22,0.12])
    elif demand == "high":
        base = np.random.choice([3,4,5,6], p=[0.20,0.35,0.30,0.15])
    elif demand == "medium":
        base = np.random.choice([2,3,4,5], p=[0.24,0.36,0.28,0.12])
    else:
        base = np.random.choice([1,2,3,4], p=[0.28,0.40,0.24,0.08])
    return min(int(base), max_possible)

for _, prod in products.iterrows():
    cat_row = categories[categories["category_id"] == prod["category_id"]].iloc[0]
    if not cat_row["rental_programme"]:
        continue  # skip non-programme categories immediately

    elig = datetime.strptime(str(prod["rental_eligible_date"]), "%Y-%m-%d")
    if elig >= PROG_END:
        continue  # product not yet eligible by programme end

    days_available = (PROG_END - elig).days

    demand = cat_row["rental_demand_tier"]
    price  = float(prod["original_retail_price"])
    stbl   = get_seasonal_table(int(prod["category_id"]), demand)
    n_rent = choose_rental_count(int(prod["category_id"]), demand, days_available)
    cur    = elig + timedelta(days=int(np.random.uniform(0, min(45, max(10, days_available//3)))))

    for _ in range(n_rent):
        if cur >= PROG_END:
            break

        month = cur.month
        if np.random.random() > min(0.98, max(0.50, 0.84 * stbl[month])):
            cur += timedelta(days=int(np.random.uniform(10, 30)))
            continue

        dur    = choose_duration(int(prod["category_id"]), demand, month)
        end_dt = cur + timedelta(days=dur)

        # Pricing model: expensive furniture → % of retail (scales with value)
        # Budget items → flat rate (simpler for lower-margin products)
        if price > 700 and np.random.random() < 0.70:
            pm = "pct_of_retail"
        elif price < 220 and np.random.random() < 0.72:
            pm = "flat_rate"
        elif np.random.random() < 0.55:
            pm = "flat_rate"
        else:
            pm = "pct_of_retail"

        eligible_rules = pricing[pricing["pricing_model"] == pm]
        eligible_rules = eligible_rules[eligible_rules["min_rental_days"] <= dur]
        if eligible_rules.empty:
            eligible_rules = pricing[pricing["pricing_model"] == pm]
        rule = eligible_rules.sample(1).iloc[0]

        base_rev = round(rule["base_daily_rate"] * dur, 2) if pm == "flat_rate" else round(rule["pct_of_retail_daily"] * price * dur, 2)

        if month in (5,6,9,11,12):
            base_rev = round(base_rev * np.random.uniform(1.03, 1.12), 2)

        is_late  = np.random.random() < np.random.uniform(0.08, 0.16)
        late_d   = int(np.random.uniform(2, 10)) if is_late else 0
        late_fee = round(rule["late_fee_per_day"] * late_d, 2) if is_late else 0.0
        ins_fee  = round(base_rev * rule["insurance_fee_pct"], 2)

        # FIX: ops cost cap lowered from 0.42 → 0.35
        # Heavy categories (delivery + assembly): 0.22–0.35
        # Standard categories (clean + transport): 0.16–0.28
        # Source: Feather ~28–32%, Furlenco ~25–30%, IKEA FaaS pilot est. 30–35%
        if int(prod["category_id"]) in [1,2,3,4,5,7,10,15]:
            base_op_pct = np.random.uniform(0.22, 0.35)
        else:
            base_op_pct = np.random.uniform(0.16, 0.28)
        op_cost = round(base_rev * base_op_pct, 2)

        total   = round(base_rev + late_fee + ins_fee, 2)
        net_rev = round(total - op_cost, 2)

        no_ret = np.random.random() < 0.025
        dbr = False
        if no_ret:
            dbr = np.random.random() < 0.50
        else:
            dbr = np.random.random() < (0.02 if int(prod["category_id"]) in [10,15] else 0.01)

        exp_ret = end_dt + timedelta(days=late_d)
        act_ret = None if no_ret else exp_ret + timedelta(days=int(np.random.choice([-2,-1,0,0,0,1,2], p=[0.02,0.05,0.60,0.15,0.08,0.07,0.03])))

        rentals_list.append({
            "rental_id":             rid,
            "product_id":            int(prod["product_id"]),
            "customer_id":           next_customer(month=month),
            "pricing_rule_id":       int(rule["rule_id"]),
            "rental_start_date":     cur.date(),
            "rental_end_date":       end_dt.date(),
            "expected_return_date":  exp_ret.date(),
            "actual_return_date":    act_ret.date() if act_ret else None,
            "rental_duration_days":  dur,
            "base_rental_revenue":   base_rev,
            "late_fee":              late_fee,
            "insurance_fee":         ins_fee,
            "total_rental_revenue":  total,
            "operational_cost":      op_cost,
            "net_rental_revenue":    net_rev,
            "is_no_return":          int(no_ret),
            "is_damaged_beyond_repair": int(dbr),
            "is_late":               int(is_late),
        })

        if not no_ret:
            if int(prod["category_id"]) in [10,15]:
                cond_probs = [0.20,0.42,0.26,0.12]
            elif int(prod["category_id"]) in [1,2,7]:
                cond_probs = [0.25,0.48,0.21,0.06]
            else:
                cond_probs = [0.30,0.46,0.18,0.06]
            cond = np.random.choice(["excellent","good","fair","damaged"], p=cond_probs)
            damage_fee = round(np.random.uniform(10, 180), 2) if cond == "damaged" and np.random.random() < 0.55 else 0.0
            returns_list.append({
                "rental_id":          rid,
                "product_id":         int(prod["product_id"]),
                "condition_on_return": cond,
                "damage_fee":         damage_fee,
                "return_note":        "",
            })

        events_list.append({
            "event_id":   rid,
            "product_id": int(prod["product_id"]),
            "event_type": "rental_start",
            "event_date": cur.date(),
            "notes":      f"rental_id={rid}",
        })

        rid += 1
        next_available = act_ret if act_ret is not None else exp_ret
        cur = next_available + timedelta(days=int(np.random.uniform(5, 20)))

rentals = pd.DataFrame(rentals_list)
returns = pd.DataFrame(returns_list)
events  = pd.DataFrame(events_list)

save("rentals", rentals)
save("return_conditions", returns)
save("inventory_events", events)


  rentals: 1,106 rows
  return_conditions: 1,079 rows
  inventory_events: 1,106 rows


## 8 · Rental Revenue vs Discount

Same central question as electronics: does rental beat markdown?

**Furniture markdown tiers — more conservative than electronics:**

IKEA items hold value unusually well because they're standardised and parts are replaceable. The AS-IS section in any IKEA store sells returned items at 20–40% off — not 50–75% like electronics.

- `slow` (Sofas, Beds, Dining, Storage, Designer): 12% off at 12mo → 45% off at 24mo+. A barely-used KIVIK sofa still holds most of its value.
- `standard` (Office, Outdoor, Lighting, Home Decor): 18% → 52%. More wear, more exposure.
- `fast` (Kids & Nursery, Event Furniture): 30% → 65%. Kids furniture is outgrown; event furniture takes heavy commercial use. Buyers are price-sensitive on both.

The comparison table is scoped to `rental_programme = True` AND eligible before `PROG_END` — same rule as electronics. Only items that were genuinely available for rental count toward the win rate.

In [9]:
def get_discount(months_unsold, dep_class):
    """
    Furniture markdown tiers — more conservative than electronics.
    IKEA items are standardised and hold value; markdowns are gradual.
    fast class: Kids & Nursery, Event Furniture (outgrown/heavy-use)
    standard: Office, Outdoor, Lighting, Home Decor
    slow: Sofas, Beds, Dining, Storage, Designer
    """
    tiers = {
        "fast":     [(12, 0.30), (18, 0.45), (24, 0.58), (999, 0.65)],
        "standard": [(12, 0.18), (18, 0.30), (24, 0.42), (999, 0.52)],
        "slow":     [(12, 0.12), (18, 0.25), (24, 0.35), (999, 0.45)],
    }
    for thr, pct in tiers[dep_class]:
        if months_unsold <= thr:
            return pct
    return tiers[dep_class][-1][1]

comparison_list = []
for _, prod in products.iterrows():
    pid    = int(prod["product_id"])
    listed = datetime.strptime(str(prod["listed_date"]), "%Y-%m-%d")
    elig   = datetime.strptime(str(prod["rental_eligible_date"]), "%Y-%m-%d")

    # FIX: only include items that were actually rental-eligible before PROG_END
    if elig >= PROG_END:
        continue

    months_unsold = (PROG_END - listed).days / 30.44
    cat_row = categories[categories["category_id"] == prod["category_id"]].iloc[0]

    if not cat_row["rental_programme"]:
        continue

    disc_pct   = get_discount(months_unsold, cat_row["depreciation_class"])
    disc_price = round(prod["original_retail_price"] * (1 - disc_pct), 2)

    prod_r  = rentals[rentals["product_id"] == pid]
    n_rents = len(prod_r)

    if n_rents > 0:
        if int(prod_r.iloc[-1]["is_damaged_beyond_repair"]) == 1 and len(prod_r) > 1:
            net_rev = round(prod_r.iloc[:-1]["net_rental_revenue"].sum(), 2)
        else:
            net_rev = round(prod_r["net_rental_revenue"].sum(), 2)
        gross_rev = round(prod_r["total_rental_revenue"].sum(), 2)
        op_cost   = round(prod_r["operational_cost"].sum(), 2)
        avg_dur   = prod_r["rental_duration_days"].mean()
        months_on = round(n_rents * avg_dur / 30.44, 2)
    else:
        net_rev = gross_rev = op_cost = months_on = 0.0
        avg_dur = 0

    ratio = round(net_rev / disc_price, 4) if disc_price > 0 else 0.0

    comparison_list.append({
        "product_id":                  pid,
        "original_retail_price":       prod["original_retail_price"],
        "months_at_enrollment":        round((elig - listed).days / 30.44, 1),
        "months_unsold_at_comparison": round(months_unsold, 1),
        "discount_pct":                disc_pct,
        "hypothetical_discount_price": disc_price,
        "total_gross_rental_revenue":  gross_rev,
        "total_operational_cost":      op_cost,
        "total_net_rental_revenue":    net_rev,
        "n_rentals":                   n_rents,
        "months_on_rental":            months_on,
        "rental_vs_discount_ratio":    ratio,
        "is_rental_more_profitable":   int(ratio > 1.0),
    })

comparison = pd.DataFrame(comparison_list)
save("rental_revenue_vs_discount", comparison)

# Quick sanity check
win_rate     = comparison["is_rental_more_profitable"].mean() * 100
median_ratio  = comparison["rental_vs_discount_ratio"].median()
mean_ratio    = comparison["rental_vs_discount_ratio"].mean()
n_with_rental = (comparison["n_rentals"] > 0).sum()
print("=" * 50)
print("DATA GENERATION SUMMARY")
print("=" * 50)
print(f"Products:              {len(products)}")
print(f"Customers:             {len(customers):,}")
print(f"Rentals:               {len(rentals):,}")
print(f"Returns:               {len(returns):,}")
print(f"Date range:            {rentals['rental_start_date'].min()} to {rentals['rental_start_date'].max()}")
print(f"Rental win rate:       {win_rate:.1f}%")
print(f"Median ratio (honest): {median_ratio:.2f}x")
print(f"Mean ratio (skewed):   {mean_ratio:.2f}x  ← inflated by early-listed products")
print(f"Items with 0 rentals:  {(comparison['n_rentals']==0).sum()}")
print("=" * 50)

comparison.head()

  rental_revenue_vs_discount: 337 rows
DATA GENERATION SUMMARY
Products:              620
Customers:             2,000
Rentals:               1,106
Returns:               1,079
Date range:            2021-01-29 to 2024-12-29
Rental win rate:       64.4%
Median ratio (honest): 2.07x
Mean ratio (skewed):   5.34x  ← inflated by early-listed products
Items with 0 rentals:  3


,product_id,original_retail_price,months_at_enrollment,months_unsold_at_comparison,discount_pct,hypothetical_discount_price,total_gross_rental_revenue,total_operational_cost,total_net_rental_revenue,n_rentals,months_on_rental,rental_vs_discount_ratio,is_rental_more_profitable
0,2,582.07,12.0,57.6,0.45,320.14,1503.14,397.54,1105.60,4,7.88,3.4535,1
1,5,298.62,12.0,25.3,0.45,164.24,1238.05,313.59,924.46,5,12.81,5.6287,1
2,6,434.57,12.0,15.7,0.25,325.93,96.97,21.33,75.64,1,2.96,0.2321,0
3,8,587.15,12.0,29.6,0.45,322.93,2216.59,653.76,1562.83,5,12.81,4.8395,1
4,9,239.07,12.0,19.2,0.35,155.40,717.90,169.88,548.02,3,4.93,3.5265,1


## 9 · Validation

Automated checks to confirm the notebook ran correctly before trusting the numbers:
- All required rental columns are present (schema match with electronics notebook)
- All 8 output CSVs exist on disk
- 365-day threshold is applied correctly to every product
- No ineligible items (eligible after PROG_END) appear in the comparison table

In [10]:
required_rentals_cols = [
    "rental_id","product_id","customer_id","pricing_rule_id",
    "rental_start_date","rental_end_date","expected_return_date","actual_return_date",
    "rental_duration_days","base_rental_revenue","late_fee","insurance_fee",
    "total_rental_revenue","operational_cost","net_rental_revenue",
    "is_no_return","is_damaged_beyond_repair","is_late"
]
missing = [c for c in required_rentals_cols if c not in rentals.columns]
assert not missing, f"Missing rentals columns: {missing}"

for fname in ["categories","products","customers","pricing_rules","rentals",
              "return_conditions","inventory_events","rental_revenue_vs_discount"]:
    path = Path(DATA_DIR) / f"{fname}.csv"
    assert path.exists(), f"Missing output file: {path}"

# Spot-check: confirm 365-day threshold is applied
sample_products = products.head(5).copy()
sample_products["days_to_eligible"] = (
    pd.to_datetime(sample_products["rental_eligible_date"]) -
    pd.to_datetime(sample_products["listed_date"])
).dt.days
assert (sample_products["days_to_eligible"] == 365).all(), "365-day threshold not applied correctly!"

# Spot-check: confirm no product in comparison table has elig >= PROG_END
comparison_pids = set(comparison["product_id"].tolist())
products_check = products[products["product_id"].isin(comparison_pids)]
late_items = products_check[pd.to_datetime(products_check["rental_eligible_date"]) >= PROG_END]
assert len(late_items) == 0, f"Found {len(late_items)} ineligible items in comparison table!"

print("Validation passed.")
print("Date range:", rentals["rental_start_date"].min(), "->", rentals["rental_start_date"].max())
print("Products:", len(products), "| Customers:", len(customers), "| Rentals:", len(rentals))
print("Comparison table rows:", len(comparison))
print("365-day threshold: ✅ confirmed")
print("No ineligible items in comparison table: ✅ confirmed")

Validation passed.
Date range: 2021-01-29 -> 2024-12-29
Products: 620 | Customers: 2000 | Rentals: 1106
Comparison table rows: 337
365-day threshold: ✅ confirmed
No ineligible items in comparison table: ✅ confirmed
